In [1]:
# Cell 1
# Mục đích: tạo lại kênh kết nối BigQuery và 3 hàm dùng chung (mb, check, stats) cho notebook mới.
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # location phải khớp vùng dataset (US)

def mb(n):
    # Đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # Dry run: chỉ ước lượng số byte sẽ quét, không chạy thật, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét khoảng: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # Chạy thật, có cầu chì max_mb; trả DataFrame kết quả
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,  # đổi MB sang byte
    )
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()  # result() chờ job xong rồi mới đọc byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [2]:
# Cell 2
# Mục đích: viết câu SQL phân loại từng order_item_id (chỉ ở lab / chỉ ở nguồn / ở cả hai) và dry run xem tốn bao nhiêu.
sql = """
SELECT
  CASE
    WHEN l.order_item_id IS NULL THEN '1. chỉ có ở nguồn hiện tại'
    WHEN s.order_item_id IS NULL THEN '2. chỉ có ở bản lab'
    WHEN l.status != s.status THEN '3. có ở cả hai, khác status'
    WHEN l.sale_price != s.sale_price THEN '4. có ở cả hai, khác giá'
    ELSE '5. có ở cả hai, giống nhau'
  END AS nhom,
  COUNT(*) AS so_dong
FROM `bq-learning-510104.dwh.fact_order_items` AS l
FULL OUTER JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
  ON l.order_item_id = s.order_item_id
GROUP BY nhom
ORDER BY nhom
"""
check(sql)

Sẽ quét khoảng: 9.08 MB


In [3]:
# Cell 3
# Mục đích: chạy thật câu ở Cell 2 và xem 5 nhóm có bao nhiêu dòng.
df = stats(sql)
print(len(df))   # in số dòng trước, để biết kết quả có rỗng không
print(df)

processed: 9.08 MB | billed: 20.0 MB
4
                          nhom  so_dong
0   1. chỉ có ở nguồn hiện tại     1175
1  3. có ở cả hai, khác status   140277
2     4. có ở cả hai, khác giá    40321
3   5. có ở cả hai, giống nhau      173


In [5]:
# Cell 4
# Mục đích: nếu nhóm 3 > 0, xem trạng thái đổi từ gì sang gì (ví dụ Processing -> Complete).
sql = """
SELECT
  l.status AS status_ban_lab,
  s.status AS status_nguon_hien_tai,
  COUNT(*) AS so_dong
FROM `bq-learning-510104.dwh.fact_order_items` AS l
JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
  ON l.order_item_id = s.order_item_id
WHERE l.status != s.status
GROUP BY status_ban_lab, status_nguon_hien_tai
ORDER BY so_dong DESC
"""
check(sql)

Sẽ quét khoảng: 6.31 MB


In [6]:
# Cell 5
# Mục đích: chạy thật Cell 4. Nếu Cell 3 cho nhóm 3 bằng 0 thì bỏ qua cell này.
df = stats(sql)
print(len(df))
print(df)

processed: 6.31 MB | billed: 20.0 MB
20
   status_ban_lab status_nguon_hien_tai  so_dong
0         Shipped              Complete    13519
1        Complete               Shipped    13419
2      Processing               Shipped    10888
3         Shipped            Processing    10760
4        Complete            Processing     9144
5      Processing              Complete     9028
6         Shipped             Cancelled     8361
7       Cancelled               Shipped     8285
8       Cancelled              Complete     6816
9        Complete             Cancelled     6789
10      Cancelled            Processing     5436
11     Processing             Cancelled     5351
12        Shipped              Returned     5340
13       Returned               Shipped     5317
14       Complete              Returned     4560
15       Returned              Complete     4544
16       Returned            Processing     3696
17     Processing              Returned     3573
18      Cancelled            

In [7]:
# Cell 5a
# Mục đích: kiểm tra trực tiếp xem cùng order_item_id ở hai bên có còn là cùng một món hàng không
# (so product_id, user_id, order_id). Nếu là cùng món thì ba cột này phải trùng gần hết.
sql = """
SELECT
  COUNT(*) AS so_dong_chung,
  COUNTIF(l.product_id = s.product_id) AS cung_product,
  COUNTIF(l.user_id = s.user_id) AS cung_user,
  COUNTIF(l.order_id = s.order_id) AS cung_order
FROM `bq-learning-510104.dwh.fact_order_items` AS l
JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
  ON l.order_item_id = s.order_item_id
"""
check(sql)

Sẽ quét khoảng: 11.07 MB


In [8]:
# Cell 5b
# Mục đích: chạy thật Cell 5a và đọc kết quả.
df = stats(sql)
print(len(df))
print(df)

processed: 11.07 MB | billed: 20.0 MB
1
   so_dong_chung  cung_product  cung_user  cung_order
0         180771             7         20         341


In [14]:
# Cell 6
# Mục đích: liệt kê các job trong 48 giờ qua có đọc fct_net_sales hoặc dim_products (không chọn user_email vì repo công khai).
sql = """
SELECT
  creation_time,
  job_type,
  statement_type,
  total_bytes_processed,
  total_bytes_billed,
  cache_hit,
  SUBSTR(query, 1, 120) AS query_dau
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 3 HOUR)
  AND EXISTS (
    SELECT 1 FROM UNNEST(referenced_tables) AS t
    WHERE t.dataset_id = 'dbt_dev' AND t.table_id IN ('fct_net_sales', 'dim_products')
  )
ORDER BY creation_time DESC
"""
check(sql)

Sẽ quét khoảng: 0.0 MB


In [15]:
# Cell 7
# Mục đích: chạy thật Cell 6 rồi đọc từng job. creation_time là UTC (giờ Việt Nam = UTC + 7).
df = stats(sql)
print(len(df))   # nếu 0 thì nới INTERVAL lên 72 HOUR trước khi kết luận
for _, row in df.iterrows():
    print(row["creation_time"], "|", row["statement_type"], "|", row["total_bytes_billed"], "|", row["query_dau"])

processed: 0.0 MB | billed: 0.0 MB
1
2026-10-11 09:01:52.033000+00:00 | SELECT | 0 | 
SELECT COUNT(*) AS so_dong
FROM `bq-learning-510104.dbt_dev.fct_net_sales`



In [13]:
# Cell 8
# Mục đích: tạo một job đối chứng (chắc chắn đọc fct_net_sales) rồi kiểm tra Cell 6 có bắt được nó không.
# Chi phí: một bảng, billed khoảng 10 MiB; chỉ đọc, không tạo hay xóa gì.
sql_ctl = """
SELECT COUNT(*) AS so_dong
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql_ctl)
df_ctl = stats(sql_ctl)
print(df_ctl)

Sẽ quét khoảng: 0.0 MB
processed: 0.0 MB | billed: 0.0 MB
   so_dong
0    45529


In [16]:
# Cell 9
# Mục đích: tạo job đối chứng có đọc dữ liệu thật (SUM đọc cột sale_price) để chắc chắn nó hiện trong JOBS nếu bộ lọc đúng.
# Chi phí: một bảng, billed khoảng 10 MiB; chỉ đọc, không tạo hay xóa gì. Chỉ chạy nếu Cell 6-7 không thấy job Cell 8.
sql_ctl = """
SELECT ROUND(SUM(sale_price), 2) AS net_revenue
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql_ctl)
df_ctl = stats(sql_ctl)
print(df_ctl)

Sẽ quét khoảng: 0.35 MB
processed: 0.35 MB | billed: 10.0 MB
   net_revenue
0   2685271.28
